# 🧪 6회차 실습: AI 에이전트 만들기

오늘은 **도구를 사용하는 AI**, 즉 에이전트를 만들어봅니다!

챗봇은 텍스트만 생성하지만, 에이전트는 **실제로 행동**할 수 있습니다.

이번 버전은 직접 `chat.completions.create(..., tools=...)` 루프를 짜는 대신, **Microsoft Agent Framework(MAF)** 가 그 루프를 맡도록 구성합니다.

In [ ]:
import json
import os
from pathlib import Path
from typing import Annotated

import matplotlib.pyplot as plt
import numpy as np
import requests
from dotenv import find_dotenv, load_dotenv
from pydantic import Field

from agent_framework import Agent, tool
from agent_framework.openai import OpenAIChatCompletionClient

load_dotenv(find_dotenv(usecwd=True))

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_KEY") or os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
MODEL = os.getenv("AZURE_OPENAI_CHAT_COMPLETION_MODEL") or os.getenv("AZURE_OPENAI_MODEL") or "gpt-4o"

if not AZURE_OPENAI_ENDPOINT or not AZURE_OPENAI_API_KEY:
    raise ValueError(".env에 AZURE_OPENAI_ENDPOINT 와 AZURE_OPENAI_KEY(또는 AZURE_OPENAI_API_KEY)를 설정해주세요.")

client = OpenAIChatCompletionClient(
    model=MODEL,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

print("✅ 준비 완료!")
print(f"MODEL = {MODEL}")

---
## 실습 1: 도구(Tool) 정의하기 🔧

AI에게 "이런 도구를 사용할 수 있어"라고 알려주는 방법을 배워봅시다.

MAF에서는 Python 함수를 `tool(...)` 로 감싸면, **함수 설명과 파라미터 스키마를 자동으로 도구로 변환**할 수 있습니다.

In [ ]:
SAFE_EVAL_GLOBALS = {
    "__builtins__": {},
    "abs": abs,
    "round": round,
    "pow": pow,
    "min": min,
    "max": max,
}

GRAPH_EVAL_GLOBALS = {
    "__builtins__": {},
    "np": np,
    "sin": np.sin,
    "cos": np.cos,
    "tan": np.tan,
    "sqrt": np.sqrt,
    "abs": np.abs,
    "pi": np.pi,
}

GRAPH_PATH = Path("graph.png")
TOOL_LOG = []


def reset_tool_log() -> None:
    TOOL_LOG.clear()


def log_tool_call(name: str, args: dict, result: str) -> None:
    TOOL_LOG.append({"name": name, "args": args, "result": result})
    print(f"🔧 도구 실행: {name}({args})")
    print(f"   📋 결과: {result}")


def calculate_raw(
    expression: Annotated[
        str,
        Field(description="계산할 수식. 예: '2+3*4', '2**10', 'round(3.14159, 2)'")
    ]
) -> str:
    """수학 수식을 계산합니다. 사칙연산, 거듭제곱 등을 지원합니다."""
    try:
        result = eval(expression, SAFE_EVAL_GLOBALS, {})
        text = str(result)
        log_tool_call("calculate", {"expression": expression}, text)
        return text
    except Exception as e:
        text = f"계산 오류: {str(e)}"
        log_tool_call("calculate", {"expression": expression}, text)
        return text


def draw_graph_raw(
    equation: Annotated[
        str,
        Field(description="그래프로 그릴 수식. x를 변수로 사용. 예: '2*x+3', 'x**2', 'sin(x)'")
    ],
    x_min: Annotated[float, Field(description="x축 최솟값 (기본값: -10)")] = -10,
    x_max: Annotated[float, Field(description="x축 최댓값 (기본값: 10)")] = 10,
) -> str:
    """수학 함수의 그래프를 그립니다. x를 변수로 사용합니다."""
    try:
        x = np.linspace(x_min, x_max, 200)
        y = eval(equation, {**GRAPH_EVAL_GLOBALS, "x": x}, {})

        plt.figure(figsize=(8, 5))
        plt.plot(x, y, linewidth=2, color="#2563eb")
        plt.title(f"y = {equation}", fontsize=14)
        plt.xlabel("x")
        plt.ylabel("y")
        plt.grid(True, alpha=0.3)
        plt.axhline(y=0, color="k", linewidth=0.5)
        plt.axvline(x=0, color="k", linewidth=0.5)
        plt.tight_layout()
        plt.savefig(GRAPH_PATH, dpi=100)
        plt.show()
        text = f"그래프가 생성되었습니다: y = {equation} (저장 위치: {GRAPH_PATH})"
        log_tool_call(
            "draw_graph",
            {"equation": equation, "x_min": x_min, "x_max": x_max},
            text,
        )
        return text
    except Exception as e:
        text = f"그래프 오류: {str(e)}"
        log_tool_call(
            "draw_graph",
            {"equation": equation, "x_min": x_min, "x_max": x_max},
            text,
        )
        return text


# 테스트
print(calculate_raw("123 * 456 + 789"))
draw_graph_raw("2*x + 3")

In [ ]:
# MAF 방식으로 도구 등록
calculate_tool = tool(
    calculate_raw,
    name="calculate",
    description="수학 수식을 계산합니다. 사칙연산, 거듭제곱 등을 지원합니다.",
)

draw_graph_tool = tool(
    draw_graph_raw,
    name="draw_graph",
    description="수학 함수의 그래프를 그립니다. x를 변수로 사용합니다.",
)

tools = [calculate_tool, draw_graph_tool]

print("✅ MAF 도구 정의 완료!")
print(f"📦 등록된 도구: {[tool_obj.name for tool_obj in tools]}")

for tool_obj in tools:
    print(f"\n### {tool_obj.name}")
    print(json.dumps(tool_obj.parameters(), ensure_ascii=False, indent=2))

---
## 실습 2: 에이전트 루프 구현 🔄

AI가 도구를 선택하고, 실행하고, 결과를 보고, 최종 답변하는 과정을 구현합니다.

이번에는 우리가 직접 반복문을 짜지 않고, **MAF의 `Agent`가 내부적으로 Think → Act → Observe → Answer 루프를 관리**합니다.

In [ ]:
math_agent = Agent(
    name="math-agent",
    client=client,
    instructions=(
        "너는 수학 문제를 풀어주는 AI 에이전트야. "
        "계산이 필요하면 calculate 도구를, 그래프가 필요하면 draw_graph 도구를 사용해. "
        "항상 친절하고 단계별로 설명해줘."
    ),
    tools=tools,
)


def extract_final_answer(response) -> str:
    assistant_messages = [message for message in response.messages if message.role == "assistant" and message.text.strip()]
    if assistant_messages:
        return assistant_messages[-1].text.strip()
    return response.text.strip()


async def run_agent(user_message: str) -> str:
    """MAF 기반 수학 에이전트를 실행합니다."""
    reset_tool_log()
    print(f"\n👤 사용자: {user_message}")
    print("=" * 50)

    response = await math_agent.run(user_message)

    if TOOL_LOG:
        print("\n🧠 MAF 내부 도구 실행 로그")
        for index, log in enumerate(TOOL_LOG, start=1):
            print(f"[{index}단계] {log['name']}({log['args']})")
            print(f"   📋 결과: {log['result']}")

    final_answer = extract_final_answer(response)
    print(f"\n🤖 최종 답변: {final_answer}")
    return final_answer

In [ ]:
# 테스트 1: 계산
await run_agent("123 곱하기 456 더하기 789는 얼마야?")

In [ ]:
# 테스트 2: 그래프
await run_agent("y = x제곱 - 4 그래프를 그려줘")

In [ ]:
# 테스트 3: 복합 질문 (계산 + 그래프)
await run_agent("y = 2x + 3에서 x가 5일 때 y값을 계산하고, 그래프도 그려줘")

### ✏️ 미니 과제: 나만의 도구 추가하기

아래 아이디어 중 하나를 골라 새로운 도구를 추가해보세요!

1. **단위 변환기**: km↔mile, kg↔lb, ℃↔℉
2. **주사위 굴리기**: 랜덤 숫자 생성
3. **날짜 계산기**: D-day 계산

In [ ]:
# 👇 여기에 새로운 도구를 만들어보세요!

def my_new_tool_raw(
    param1: Annotated[str, Field(description="새 도구에 전달할 입력")]
) -> str:
    """새 도구 설명"""
    return f"결과: {param1}"


my_new_tool = tool(
    my_new_tool_raw,
    name="my_new_tool",
    description="새 도구 설명",
)

# 기존 도구 목록에 추가해서 새 에이전트를 만들면 됩니다.
custom_tools = tools + [my_new_tool]
print([tool_obj.name for tool_obj in custom_tools])

# 예시:
# custom_agent = Agent(
#     name="custom-agent",
#     client=client,
#     instructions="너는 새 도구도 쓸 수 있는 에이전트야.",
#     tools=custom_tools,
# )

---
## 실습 3: 웹 검색 에이전트 🌐

인터넷에서 정보를 검색하는 도구를 가진 에이전트를 만들어봅시다.

이번에도 검색 함수 자체는 직접 만들고, 에이전트 루프는 MAF가 담당합니다.

In [ ]:
def web_search_raw(
    query: Annotated[str, Field(description="위키피디아에서 검색할 키워드")]
) -> str:
    """웹에서 정보를 검색합니다 (간단한 위키피디아 검색)."""
    try:
        safe_query = requests.utils.quote(query)
        url = f"https://ko.wikipedia.org/api/rest_v1/page/summary/{safe_query}"
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            text = data.get("extract", "검색 결과를 찾을 수 없습니다.")
        else:
            text = "검색 결과를 찾을 수 없습니다."
    except Exception as e:
        text = f"검색 오류: {str(e)}"

    log_tool_call("web_search", {"query": query}, text)
    return text


web_search_tool = tool(
    web_search_raw,
    name="web_search",
    description="웹에서 정보를 검색합니다 (간단한 위키피디아 검색).",
)

print(web_search_raw("인공지능"))

In [ ]:
research_agent = Agent(
    name="research-agent",
    client=client,
    instructions=(
        "너는 웹 검색 도구를 사용할 수 있는 정보 탐색 에이전트야. "
        "필요할 때 web_search 도구를 사용하고, 검색 결과를 짧고 정확하게 정리해줘."
    ),
    tools=[web_search_tool],
)


async def run_web_agent(user_message: str) -> str:
    reset_tool_log()
    print(f"\n👤 사용자: {user_message}")
    print("=" * 50)

    response = await research_agent.run(user_message)

    if TOOL_LOG:
        print("\n🧠 MAF 내부 도구 실행 로그")
        for index, log in enumerate(TOOL_LOG, start=1):
            print(f"[{index}단계] {log['name']}({log['args']})")
            print(f"   📋 결과: {log['result']}")

    final_answer = extract_final_answer(response)
    print(f"\n🤖 최종 답변: {final_answer}")
    return final_answer

In [ ]:
# 테스트: 웹 검색 에이전트
await run_web_agent("인공지능이 무엇인지 한 문단으로 설명해줘")

---
## 🎉 수고하셨습니다!

오늘 배운 내용:
- **Function Calling**: AI가 도구를 선택하고 호출하는 방법
- **에이전트 루프**: Think → Act → Observe → Answer
- **도구 만들기**: 계산기, 그래프, 웹 검색
- **Microsoft Agent Framework**: 직접 루프를 짜지 않고 Agent와 Tool로 구성하는 방법

### 📝 숙제: 꼬맨틀(Semantle) 클론 만들기!
`homework/` 폴더를 확인하세요.